# 🌸 LILY HUNYUAN 1.5 FAST I2V — KAGGLE (XET-FIXED)

Image → video with HunyuanVideo‑1.5 480p I2V step-distilled. This revision avoids the Kaggle/Hugging Face `Background writer channel closed` Xet failure by forcing normal HTTP download, one file at a time, with retries.

**IMPORTANT:** after importing this updated notebook into Kaggle, start a **fresh session**, enable **GPU + Internet**, then **Run All**. Start with **49 frames + 8 steps**.


In [ ]:
# CELL 1 — MUST RUN FIRST. Set HF download behavior BEFORE any Hugging Face import.
import os, sys, subprocess

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_XET_RECONSTRUCT_WRITE_SEQUENTIALLY"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "3600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/kaggle/working/hf-cache"
os.environ["HF_HUB_CACHE"] = "/kaggle/working/hf-cache/hub"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

packages = [
    "git+https://github.com/huggingface/diffusers.git@c5469b7ceb606edd7ba6570dcd17d38590a18db6",
    "transformers>=4.57.0",
    "accelerate>=1.10.0",
    "huggingface_hub>=0.34.0",
    "safetensors>=0.5.0",
    "sentencepiece",
    "protobuf",
    "gradio>=5.45,<7",
    "imageio[ffmpeg]",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])

# huggingface_hub can auto-use hf-xet if present. Remove it so Kaggle cannot fall back into the broken Xet path.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "hf-xet"], check=False, stdout=subprocess.DEVNULL)

# Extra protection if Kaggle happened to pre-import HF in this kernel.
if "huggingface_hub.constants" in sys.modules:
    import huggingface_hub.constants as _hfc
    _hfc.HF_HUB_DISABLE_XET = True

print("✅ Dependencies ready")
print("✅ Xet disabled; model will use standard HTTP downloads")


In [ ]:
# CELL 2 — GPU sanity check
import os, gc, time, warnings, torch
from PIL import Image

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Kaggle → Settings → Accelerator → GPU, then restart session.")

gpu_name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() and cc[0] >= 8 else torch.float16

print(f"✅ GPU: {gpu_name} | VRAM: {vram:.1f} GB | dtype: {DTYPE}")
print(f"✅ GPUs visible: {torch.cuda.device_count()}")


In [ ]:
# CELL 3 — DOWNLOAD MODEL SAFELY FIRST (serial + retry), then load locally
import os, time, gc, shutil
from huggingface_hub import snapshot_download

MODEL_ID = "hunyuanvideo-community/HunyuanVideo-1.5-Diffusers-480p_i2v_step_distilled"
CACHE_DIR = "/kaggle/working/hf-cache"

print("⬇️ Downloading HunyuanVideo 1.5 weights...")
print("This first download is large. If Kaggle briefly drops the connection, the retry resumes cached files instead of starting over.")

snapshot_path = None
last_error = None
for attempt in range(1, 4):
    try:
        print(f"\nDownload attempt {attempt}/3...")
        snapshot_path = snapshot_download(
            repo_id=MODEL_ID,
            cache_dir=CACHE_DIR,
            max_workers=1,
            resume_download=True,
        )
        break
    except Exception as e:
        last_error = e
        print(f"⚠️ Attempt {attempt} failed: {type(e).__name__}: {e}")
        if attempt < 3:
            print("Waiting 15 seconds, then resuming...")
            time.sleep(15)

if snapshot_path is None:
    raise RuntimeError(f"Model download failed after 3 attempts: {last_error}")

print("✅ MODEL DOWNLOAD COMPLETE")
print("Snapshot:", snapshot_path)
print("Free disk GB:", round(shutil.disk_usage('/kaggle/working').free/1024**3, 1))


In [ ]:
# CELL 4 — LOAD ONCE. All later generations reuse this same pipeline.
import gc, warnings, torch
from diffusers import HunyuanVideo15ImageToVideoPipeline
from diffusers.hooks import apply_group_offloading

print("🌸 Loading downloaded model into the pipeline...")
pipe = HunyuanVideo15ImageToVideoPipeline.from_pretrained(
    snapshot_path,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    local_files_only=True,
)

pipe.vae.enable_tiling()
try:
    pipe.vae.enable_slicing()
except Exception:
    pass

cuda = torch.device("cuda:0")
cpu = torch.device("cpu")

try:
    pipe.transformer.enable_group_offload(
        onload_device=cuda, offload_device=cpu,
        offload_type="block_level", num_blocks_per_group=2,
        use_stream=True, non_blocking=True,
    )
    print("✅ Transformer group-offload enabled")
except Exception as e:
    print("⚠️ Block offload unavailable; using leaf offload:", e)
    pipe.transformer.enable_group_offload(
        onload_device=cuda, offload_device=cpu,
        offload_type="leaf_level", use_stream=False,
    )

for name in ("text_encoder", "text_encoder_2"):
    module = getattr(pipe, name, None)
    if module is None:
        continue
    try:
        apply_group_offloading(
            module, onload_device=cuda, offload_device=cpu,
            offload_type="block_level", num_blocks_per_group=4,
            use_stream=True, non_blocking=True,
        )
    except Exception:
        apply_group_offloading(
            module, onload_device=cuda, offload_device=cpu,
            offload_type="leaf_level", use_stream=False,
        )

for name in ("vae", "image_encoder"):
    module = getattr(pipe, name, None)
    if module is not None:
        try:
            module.to(cuda)
        except Exception as e:
            print(f"⚠️ {name} left off-GPU: {e}")

gc.collect(); torch.cuda.empty_cache()
print("✅ MODEL READY — later generations do NOT reload it")
print(f"Allocated VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


In [ ]:
# CELL 5 — iPhone-friendly generator UI
import os, time, uuid, gc, traceback, torch, gradio as gr
from diffusers.utils import export_to_video

OUTPUT_DIR = "/kaggle/working/lily_hunyuan_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PRESETS = {
    "⚡ TEST — 49 frames (~2 sec)": 49,
    "🌸 BALANCED — 73 frames (~3 sec)": 73,
    "🎬 FULL — 121 frames (~5 sec)": 121,
}

def generate_video(image, prompt, negative_prompt, preset, steps, seed, fps):
    if image is None:
        raise gr.Error("Upload a starting image first.")
    if not prompt or not prompt.strip():
        raise gr.Error("Write a motion prompt first.")

    image = image.convert("RGB")
    frames = PRESETS[preset]
    steps = int(steps)
    seed = int(seed)
    fps = int(fps)
    out = os.path.join(OUTPUT_DIR, f"lily_hy15_{int(time.time())}_{uuid.uuid4().hex[:6]}.mp4")
    gen = torch.Generator(device="cuda").manual_seed(seed)

    kwargs = dict(
        image=image, prompt=prompt.strip(), num_frames=frames,
        num_inference_steps=steps, generator=gen,
    )
    if negative_prompt and negative_prompt.strip():
        kwargs["negative_prompt"] = negative_prompt.strip()

    started = time.time()
    try:
        with torch.inference_mode():
            video = pipe(**kwargs).frames[0]
        export_to_video(video, out, fps=fps)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        raise gr.Error("GPU out of memory. Restart session and try TEST / 49 frames / 8 steps first.")
    except Exception as e:
        gc.collect(); torch.cuda.empty_cache(); traceback.print_exc()
        raise gr.Error(f"Generation failed: {type(e).__name__}: {e}")

    seconds = time.time() - started
    del video
    gc.collect(); torch.cuda.empty_cache()
    return out, f"✅ {seconds:.0f}s • {frames} frames • {steps} steps • seed {seed} • model remains loaded"

with gr.Blocks(title="Lily Hunyuan 1.5 Studio") as demo:
    gr.Markdown("# 🌸 Lily Hunyuan 1.5 Studio\n**FIRST TEST: 49 frames + 8 steps.**")
    image_in = gr.Image(type="pil", label="Starting image")
    prompt_in = gr.Textbox(label="Motion / scene prompt", lines=5, placeholder="Natural movement, subtle handheld camera...")
    negative_in = gr.Textbox(label="Negative prompt (optional)", lines=2, value="blurry, distorted anatomy, jitter, flicker, low quality")
    preset_in = gr.Dropdown(list(PRESETS.keys()), value=list(PRESETS.keys())[0], label="Length")
    steps_in = gr.Radio([8, 12], value=8, label="Steps")
    seed_in = gr.Number(value=12345, precision=0, label="Seed")
    fps_in = gr.Slider(12, 30, value=24, step=1, label="Playback FPS")
    btn = gr.Button("🌸 GENERATE VIDEO", variant="primary")
    video_out = gr.Video(label="Result")
    status_out = gr.Textbox(label="Status")
    btn.click(
        generate_video,
        [image_in, prompt_in, negative_in, preset_in, steps_in, seed_in, fps_in],
        [video_out, status_out],
    )

demo.queue(max_size=2).launch(share=True, debug=False)
